# 交互式查询本地 DuckLake (`fsc_lake`)

只读连接到项目的 DuckLake catalog（元数据在**本机 Postgres** 库 `fsc_lake`，数据文件在 `./lake_pg`），方便随手 SQL 探查 bronze/silver/gold。

**运行方式**：在 VS Code 里直接打开本文件，右上角选 kernel 为 `/home/ubuntu/.venv`（已有 `ipykernel`+`polars`+`duckdb 1.5.4`），逐格运行即可。无需安装 jupyterlab。

**要点**
- 默认 `READ_ONLY=True`：只读连接。catalog 是 Postgres 后端（支持并发写），只读查询不会干扰 Dagster/dbt。想写入改 `False`。
- 需 duckdb **>= 1.5**（DuckLake 1.0 格式；旧版报 `Only DuckLake versions ...0.3 are supported`）。
- 连接串（含密码）由 `catalog.py` 从 `local.setting.json` 的 `catalog_pg` 块构建，笔记本里不出现明文。
- 所有表在 `lake.<schema>.<table>`；连接时已 `USE lake.bronze`，所以 bronze 表可直接写表名。

In [1]:
import duckdb
import polars as pl
import sys

# 用项目里的 catalog.py 构建连接（catalog 元数据在本机 Postgres 库 fsc_lake，
# 免手写密码；数据文件是 ./lake_pg 下的 Parquet）
sys.path.insert(0, "/home/ubuntu/dagster")
import catalog

READ_ONLY = True  # 只读。Postgres 后端支持并发，只读查询不会干扰 Dagster/dbt


def connect(read_only: bool = READ_ONLY) -> duckdb.DuckDBPyConnection:
    """打开一个已 attach 共享 DuckLake（Postgres catalog）的 duckdb 连接。"""
    con = duckdb.connect()
    con.execute("INSTALL postgres; LOAD postgres; INSTALL ducklake; LOAD ducklake;")
    ro = " (READ_ONLY)" if read_only else ""
    # 连接由 catalog.py 从 local.setting.json 的 catalog_pg 块构建
    con.execute(f"ATTACH '{catalog.duckdb_attach_target()}' AS lake{ro}")
    con.execute("USE lake.bronze")  # 默认 schema，bronze 表可不带前缀
    return con


con = connect()
print("duckdb", duckdb.__version__, "| catalog: postgres fsc_lake | read_only=", READ_ONLY)

duckdb 1.5.4 | catalog: postgres fsc_lake | read_only= True


In [2]:
def q(sql: str) :
    """跑 SQL 返回 pandas DataFrame。bronze 表可直接写表名，
    其它层用全名，如 lake.silver.xxx / lake.gold.xxx。
    例: q(\"SELECT * FROM accounts LIMIT 5\")"""
    return con.execute(sql).pl()

## 1. 有哪些表

In [3]:
q("""
SELECT table_schema, table_name
FROM information_schema.tables
WHERE table_catalog = 'lake'
  AND table_schema NOT LIKE '%staging%'   -- 隐藏 dlt 的 *_staging
ORDER BY 1, 2
""")

table_schema,table_name
str,str
"""bronze""","""_dlt_loads"""
"""bronze""","""_dlt_pipeline_state"""
"""bronze""","""_dlt_version"""
"""bronze""","""accounts"""
"""bronze""","""fsc_aaf_calculationses"""
"""bronze""","""recordtype"""
"""silver""","""accounts"""
"""silver""","""recordtype"""


## 2. 各 bronze 表的行数

In [4]:
tables = q("""
    SELECT table_name FROM information_schema.tables
    WHERE table_catalog='lake' AND table_schema='bronze'
      AND table_name NOT LIKE '\\_dlt%' ESCAPE '\\'   -- 跳过 dlt 内部表
    ORDER BY 1
""")["table_name"].to_list()

rows = [
    {"table": t, "rows": con.execute(f'SELECT count(*) FROM bronze."{t}"').fetchone()[0]}
    for t in tables
]
pl.DataFrame(rows)

table,rows
str,i64
"""accounts""",1026
"""fsc_aaf_calculationses""",1530334
"""recordtype""",46


## 3. 预览某张表

In [5]:
con.execute("SELECT * FROM fsc_aaf_calculationses LIMIT 20").pl()

@odata.etag,_fsc_aaf_projectcertificate_value@OData.Community.Display.V1.FormattedValue,_fsc_aaf_projectcertificate_value@Microsoft.Dynamics.CRM.associatednavigationproperty,_fsc_aaf_projectcertificate_value@Microsoft.Dynamics.CRM.lookuplogicalname,_fsc_aaf_projectcertificate_value,statuscode@OData.Community.Display.V1.FormattedValue,statuscode,fsc_aaf_sfid,fsc_aaf_quarter@OData.Community.Display.V1.FormattedValue,fsc_aaf_quarter,fsc_aaf_isrecordcreated@OData.Community.Display.V1.FormattedValue,fsc_aaf_isrecordcreated,_fsc_aaf_country_value@OData.Community.Display.V1.FormattedValue,_fsc_aaf_country_value@Microsoft.Dynamics.CRM.associatednavigationproperty,_fsc_aaf_country_value@Microsoft.Dynamics.CRM.lookuplogicalname,_fsc_aaf_country_value,fsc_aaf_pcstatus@OData.Community.Display.V1.FormattedValue,fsc_aaf_pcstatus,fsc_aaf_pcname,_ownerid_value@OData.Community.Display.V1.FormattedValue,_ownerid_value@Microsoft.Dynamics.CRM.associatednavigationproperty,_ownerid_value@Microsoft.Dynamics.CRM.lookuplogicalname,_ownerid_value,fsc_aaf_triggerprocess@OData.Community.Display.V1.FormattedValue,fsc_aaf_triggerprocess,fsc_aaf_cb,_fsc_aaf_batch_value@OData.Community.Display.V1.FormattedValue,_fsc_aaf_batch_value@Microsoft.Dynamics.CRM.associatednavigationproperty,_fsc_aaf_batch_value@Microsoft.Dynamics.CRM.lookuplogicalname,_fsc_aaf_batch_value,fsc_aaf_pccode,_fsc_aaf_method_value@OData.Community.Display.V1.FormattedValue,_fsc_aaf_method_value@Microsoft.Dynamics.CRM.associatednavigationproperty,_fsc_aaf_method_value@Microsoft.Dynamics.CRM.lookuplogicalname,_fsc_aaf_method_value,fsc_aaf_isaafexempted@OData.Community.Display.V1.FormattedValue,fsc_aaf_isaafexempted,…,fsc_aaf_area_plantation@OData.Community.Display.V1.FormattedValue,fsc_aaf_area_plantation,fsc_aaf_area_slimfconversation@OData.Community.Display.V1.FormattedValue,fsc_aaf_area_slimfconversation,fsc_aaf_area_slimftropical@OData.Community.Display.V1.FormattedValue,fsc_aaf_area_slimftropical,fsc_aaf_area_slimftemperate@OData.Community.Display.V1.FormattedValue,fsc_aaf_area_slimftemperate,fsc_aaf_area_community@OData.Community.Display.V1.FormattedValue,fsc_aaf_area_community,fsc_aaf_area_slimfboreal@OData.Community.Display.V1.FormattedValue,fsc_aaf_area_slimfboreal,fsc_aaf_area_slimfcommunity@OData.Community.Display.V1.FormattedValue,fsc_aaf_area_slimfcommunity,fsc_aaf_processoraafclass@OData.Community.Display.V1.FormattedValue,fsc_aaf_processoraafclass,fsc_aaf_traderaafclass@OData.Community.Display.V1.FormattedValue,fsc_aaf_traderaafclass,fsc_aaf_fpt_processor,fsc_aaf_totalrevenue,fsc_aafassumedprocessorclass,fsc_aafassumedprocessorturnover,fsc_aaf_fpt_trader,fsc_aafassumedtraderturnover,fsc_aafassumedtraderclass,fsc_snapshotstatus@OData.Community.Display.V1.FormattedValue,fsc_snapshotstatus,fsc_calculation,fsc_aaf_isbackdatedcertificate@OData.Community.Display.V1.FormattedValue,fsc_aaf_isbackdatedcertificate,_modifiedonbehalfby_value@OData.Community.Display.V1.FormattedValue,_modifiedonbehalfby_value@Microsoft.Dynamics.CRM.lookuplogicalname,_modifiedonbehalfby_value,fsc_scope,fsc_reasonforsuspensiontermination@OData.Community.Display.V1.FormattedValue,fsc_reasonforsuspensiontermination,fsc_fsccomments
str,str,str,str,str,str,i64,str,str,i64,str,bool,str,str,str,str,str,i64,str,str,str,str,str,str,bool,str,str,str,str,str,str,str,str,str,str,str,bool,…,str,f64,str,f64,str,f64,str,f64,str,f64,str,f64,str,f64,str,i64,str,i64,str,str,str,str,str,str,str,str,str,str,str,bool,str,str,str,str,str,i64,str
"""W/""472597344""""",null,null,null,null,"""Active""",1,"""https://na151.salesforce.com/a…",null,null,"""Yes""",true,"""SPAIN""","""fsc_aaf_country""","""fsc_countrydata""","""651f8543-a036-ea11-a812-000d3a…",null,null,null,"""CRM Administrator""","""ownerid""","""systemuser""","""80c38cd3-5905-ea11-a812-000d3a…","""Yes""",true,"""BMC""","""2023 - BMC - Q1""","""fsc_aaf_batch""","""fsc_aaf_batch""","""203c886b-2389-ed11-81ad-000d3a…",null,"""AAFFEE - 1002""","""fsc_aaf_method""","""fsc_

In [6]:
q("SELECT * FROM fsc_aaf_calculationses LIMIT 20")

@odata.etag,_fsc_aaf_projectcertificate_value@OData.Community.Display.V1.FormattedValue,_fsc_aaf_projectcertificate_value@Microsoft.Dynamics.CRM.associatednavigationproperty,_fsc_aaf_projectcertificate_value@Microsoft.Dynamics.CRM.lookuplogicalname,_fsc_aaf_projectcertificate_value,statuscode@OData.Community.Display.V1.FormattedValue,statuscode,fsc_aaf_sfid,fsc_aaf_quarter@OData.Community.Display.V1.FormattedValue,fsc_aaf_quarter,fsc_aaf_isrecordcreated@OData.Community.Display.V1.FormattedValue,fsc_aaf_isrecordcreated,_fsc_aaf_country_value@OData.Community.Display.V1.FormattedValue,_fsc_aaf_country_value@Microsoft.Dynamics.CRM.associatednavigationproperty,_fsc_aaf_country_value@Microsoft.Dynamics.CRM.lookuplogicalname,_fsc_aaf_country_value,fsc_aaf_pcstatus@OData.Community.Display.V1.FormattedValue,fsc_aaf_pcstatus,fsc_aaf_pcname,_ownerid_value@OData.Community.Display.V1.FormattedValue,_ownerid_value@Microsoft.Dynamics.CRM.associatednavigationproperty,_ownerid_value@Microsoft.Dynamics.CRM.lookuplogicalname,_ownerid_value,fsc_aaf_triggerprocess@OData.Community.Display.V1.FormattedValue,fsc_aaf_triggerprocess,fsc_aaf_cb,_fsc_aaf_batch_value@OData.Community.Display.V1.FormattedValue,_fsc_aaf_batch_value@Microsoft.Dynamics.CRM.associatednavigationproperty,_fsc_aaf_batch_value@Microsoft.Dynamics.CRM.lookuplogicalname,_fsc_aaf_batch_value,fsc_aaf_pccode,_fsc_aaf_method_value@OData.Community.Display.V1.FormattedValue,_fsc_aaf_method_value@Microsoft.Dynamics.CRM.associatednavigationproperty,_fsc_aaf_method_value@Microsoft.Dynamics.CRM.lookuplogicalname,_fsc_aaf_method_value,fsc_aaf_isaafexempted@OData.Community.Display.V1.FormattedValue,fsc_aaf_isaafexempted,…,fsc_aaf_area_plantation@OData.Community.Display.V1.FormattedValue,fsc_aaf_area_plantation,fsc_aaf_area_slimfconversation@OData.Community.Display.V1.FormattedValue,fsc_aaf_area_slimfconversation,fsc_aaf_area_slimftropical@OData.Community.Display.V1.FormattedValue,fsc_aaf_area_slimftropical,fsc_aaf_area_slimftemperate@OData.Community.Display.V1.FormattedValue,fsc_aaf_area_slimftemperate,fsc_aaf_area_community@OData.Community.Display.V1.FormattedValue,fsc_aaf_area_community,fsc_aaf_area_slimfboreal@OData.Community.Display.V1.FormattedValue,fsc_aaf_area_slimfboreal,fsc_aaf_area_slimfcommunity@OData.Community.Display.V1.FormattedValue,fsc_aaf_area_slimfcommunity,fsc_aaf_processoraafclass@OData.Community.Display.V1.FormattedValue,fsc_aaf_processoraafclass,fsc_aaf_traderaafclass@OData.Community.Display.V1.FormattedValue,fsc_aaf_traderaafclass,fsc_aaf_fpt_processor,fsc_aaf_totalrevenue,fsc_aafassumedprocessorclass,fsc_aafassumedprocessorturnover,fsc_aaf_fpt_trader,fsc_aafassumedtraderturnover,fsc_aafassumedtraderclass,fsc_snapshotstatus@OData.Community.Display.V1.FormattedValue,fsc_snapshotstatus,fsc_calculation,fsc_aaf_isbackdatedcertificate@OData.Community.Display.V1.FormattedValue,fsc_aaf_isbackdatedcertificate,_modifiedonbehalfby_value@OData.Community.Display.V1.FormattedValue,_modifiedonbehalfby_value@Microsoft.Dynamics.CRM.lookuplogicalname,_modifiedonbehalfby_value,fsc_scope,fsc_reasonforsuspensiontermination@OData.Community.Display.V1.FormattedValue,fsc_reasonforsuspensiontermination,fsc_fsccomments
str,str,str,str,str,str,i64,str,str,i64,str,bool,str,str,str,str,str,i64,str,str,str,str,str,str,bool,str,str,str,str,str,str,str,str,str,str,str,bool,…,str,f64,str,f64,str,f64,str,f64,str,f64,str,f64,str,f64,str,i64,str,i64,str,str,str,str,str,str,str,str,str,str,str,bool,str,str,str,str,str,i64,str
"""W/""472597344""""",null,null,null,null,"""Active""",1,"""https://na151.salesforce.com/a…",null,null,"""Yes""",true,"""SPAIN""","""fsc_aaf_country""","""fsc_countrydata""","""651f8543-a036-ea11-a812-000d3a…",null,null,null,"""CRM Administrator""","""ownerid""","""systemuser""","""80c38cd3-5905-ea11-a812-000d3a…","""Yes""",true,"""BMC""","""2023 - BMC - Q1""","""fsc_aaf_batch""","""fsc_aaf_batch""","""203c886b-2389-ed11-81ad-000d3a…",null,"""AAFFEE - 1002""","""fsc_aaf_method""","""fsc_

In [7]:
import polars as pl

In [8]:
# 需要脱离 catalog 拿 Parquet 时，用 catalog 导出「一致视图」，
# 不要直接 glob ./lake_pg 原始文件（含多快照 + schema 演进，会重复/列不齐）。
con.execute("COPY (SELECT * FROM recordtype) TO '/tmp/recordtype.parquet' (FORMAT parquet)")
pl.read_parquet("/tmp/recordtype.parquet")

Id,Name,DeveloperName,NamespacePrefix,Description,BusinessProcessId,SobjectType,IsActive,CreatedById,CreatedDate,LastModifiedById,LastModifiedDate,SystemModstamp
str,str,str,str,str,str,str,str,str,str,str,str,"datetime[μs, UTC]"
"""012400000005VqaAAE""","""Dispute""","""Dispute""",null,"""For use with dispute resolutio…","""019400000008p82AAA""","""Case""","""true""","""005400000012ACIAA2""","""2010-01-19T09:13:45.000Z""","""005400000012ACIAA2""","""2010-01-20T10:44:44.000Z""",2010-01-20 10:44:44 UTC
"""01240000000IPDsAAO""","""Key Account""","""Key_Account""",null,null,"""019400000000NGcAAM""","""Lead""","""true""","""00540000001Ee9cAAC""","""2012-12-04T16:50:34.000Z""","""00540000001Ee9cAAC""","""2013-05-07T09:35:20.000Z""",2013-05-07 09:35:20 UTC
"""01240000000DPEgAAO""","""Online Donation""","""OnlineDonation""",null,null,"""019400000000NGcAAM""","""Lead""","""true""","""00540000000wj3zAAA""","""2008-04-26T19:54:14.000Z""","""0054000000263EtAAI""","""2016-06-24T09:32:53.000Z""",2016-06-24 09:32:53 UTC
"""01233000000DxJgAAK""","""Earned""","""Earned""",null,"""Earned Revenue from services""","""019400000000NGdAAM""","""Opportunity""","""true""","""0054000000263EtAAI""","""2016-11-11T10:37:54.000Z""","""0054000000263EtAAI""","""2022-07-07T15:20:23.000Z""",2022-07-07 15:20:23 UTC
"""01233000000Dx9CAAS""","""Council""","""Council""",null,"""World Forest Council / advisor…","""019400000000NGdAAM""","""Opportunity""","""true""","""0054000000263EtAAI""","""2016-05-19T11:28:40.000Z""","""0054000000263EtAAI""","""2022-07-07T15:20:23.000Z""",2022-07-07 15:20:23 UTC
…,…,…,…,…,…,…,…,…,…,…,…,…
"""01240000000HoZwAAK""","""COC""","""COC""",null,"""Chain of Custody""",null,"""Certificate_c__c""","""True""","""005400000012ACIAA2""","""1257774979000""","""005Do000002EoRjIAK""","""1782986654000""",2026-07-02 10:04:14 UTC
"""01240000000HoaHAAS""","""CW/FM""","""CW_FM""",null,"""Controlled Wood / Forest Mana…",null,"""Certificate_c__c""","""True""","""005400000012ACIAA2""","""1257774979000""","""005Do000002EoRjIAK""","""1782986654000""",2026-07-02 10:04:14 UTC
"""01240000000HoaIAAS""","""FM""","""FM""",null,"""Forest Management""",null,"""Certificate_c__c""","""True""","""005400000012ACIAA2""","""1257774979000""","""005Do000002EoRjIAK""","""1782986654000""",2026-07-02 10:04:14 UTC


## 4. 看某张表的列 / 类型

In [9]:
q("""
SELECT column_name, data_type
FROM information_schema.columns
WHERE table_catalog='lake' AND table_schema='bronze'
  AND table_name = 'fsc_aaf_calculationses'
ORDER BY ordinal_position
""")

column_name,data_type
str,str
"""@odata.etag""","""VARCHAR"""
"""_fsc_aaf_projectcertificate_va…","""VARCHAR"""
"""_fsc_aaf_projectcertificate_va…","""VARCHAR"""
"""_fsc_aaf_projectcertificate_va…","""VARCHAR"""
"""_fsc_aaf_projectcertificate_va…","""VARCHAR"""
…,…
"""_modifiedonbehalfby_value""","""VARCHAR"""
"""fsc_scope""","""VARCHAR"""
"""fsc_reasonforsuspensiontermina…","""VARCHAR"""


## 5. 你自己的查询（scratch）

In [10]:
q("""
-- 在这里写你的 SQL
SELECT count(*) AS n
FROM fsc_aaf_calculationses
""")

n
i64
1530334


## 6. DuckLake 特性：快照 & 时间旅行

DuckLake 每次写入都产生一个 snapshot，可以查历史版本。

In [11]:
# 最近 10 个快照
q("SELECT snapshot_id, snapshot_time, schema_version FROM ducklake_snapshots('lake') ORDER BY snapshot_id DESC LIMIT 10")

snapshot_id,snapshot_time,schema_version
i64,"datetime[μs, Europe/Berlin]",i64
12,2026-07-15 13:06:07.836180 CEST,10
11,2026-07-15 13:06:07.596509 CEST,9
10,2026-07-15 13:06:07.513145 CEST,8
9,2026-07-15 13:05:02.666765 CEST,7
8,2026-07-15 13:05:02.653548 CEST,7
7,2026-07-15 12:56:59.397854 CEST,7
6,2026-07-15 12:46:35.061038 CEST,6
5,2026-07-15 12:46:34.954250 CEST,5
4,2026-07-15 12:46:34.788112 CEST,4


In [12]:
# 时间旅行：读某个历史快照时的表内容（把 42 换成上面的 snapshot_id）
# q("SELECT count(*) FROM accounts AT (VERSION => 42)")

## 7. 收尾

In [13]:
con.close()
print("connection closed")

connection closed
